# 05 — SMOTE-ENN (hanya pada training set)

Notebook ini menerapkan teknik hybrid resampling **SMOTE-ENN** untuk menangani
ketidakseimbangan kelas (mild imbalance 40:60) sekaligus membersihkan noise akibat
class overlapping.

**Aturan kritis anti-data-leakage (AGENTS.md):**

- SMOTE-ENN **HANYA** diterapkan pada `X_train`/`y_train`.
- `X_test`/`y_test` **tidak disentuh sama sekali** — divalidasi eksplisit dengan
  `assert` bahwa ukuran & isinya tidak berubah.

Urutan yang benar: split → SMOTE-ENN pada training → (latih model di notebook
berikutnya) → prediksi pada test set murni.

In [1]:
import os
import pandas as pd
import numpy as np
from joblib import load, dump

from imblearn.combine import SMOTEENN

RANDOM_STATE = 42
DATA_DIR = "../data/processed"

In [2]:
X_train = load(os.path.join(DATA_DIR, "X_train.joblib"))
X_test = load(os.path.join(DATA_DIR, "X_test.joblib"))
y_train = load(os.path.join(DATA_DIR, "y_train.joblib"))
y_test = load(os.path.join(DATA_DIR, "y_test.joblib"))

print("X_train", X_train.shape, "| y_train", y_train.shape)
print("X_test ", X_test.shape, "| y_test ", y_test.shape)

X_train (32051, 4) | y_train (32051,)
X_test  (8013, 4) | y_test  (8013,)


## 1. Distribusi Kelas Sebelum SMOTE-ENN

In [3]:
def show_dist(title, y):
    s = pd.Series(y)
    counts = s.value_counts().sort_index()
    pct = (s.value_counts(normalize=True).sort_index() * 100).round(2)
    print(f"{title} (total {len(s):,}):")
    for cls in [0, 1]:
        label = "Not Stunted" if cls == 0 else "Stunted"
        print(f"    {label} ({cls}): {counts.get(cls, 0):,}  ({pct.get(cls, 0):.2f}%)")
    print()


show_dist("y_train SEBELUM SMOTE-ENN", y_train)

y_train SEBELUM SMOTE-ENN (total 32,051):
    Not Stunted (0): 19,091  (59.56%)
    Stunted (1): 12,960  (40.44%)



## 2. Terapkan SMOTE-ENN (hanya pada training set)

`SMOTEENN` = **SMOTE** (oversampling sintetis kelas minoritas, default `k_neighbors=5`)
diikuti **ENN/EditedNearestNeighbours** (pembersih noise, default `n_neighbors=3`).
`random_state=42` untuk reproducibility.

**Validasi eksplisit** bahwa test set tidak berubah (jumlah baris & isi).

In [4]:
n_test_before = len(X_test)
X_test_snapshot = X_test.copy(deep=True)
y_test_snapshot = y_test.copy(deep=True)

smote_enn = SMOTEENN(random_state=RANDOM_STATE)
X_train_resampled, y_train_resampled = smote_enn.fit_resample(X_train, y_train)

# Validasi anti-leakage: test set tidak boleh berubah
assert len(X_test) == n_test_before, "X_test berubah jumlah barisnya! Jangan resample test set."
assert X_test.equals(X_test_snapshot), "X_test termodifikasi isinya!"
assert y_test.equals(y_test_snapshot), "y_test termodifikasi!"
print("Validasi lulus: X_test & y_test tidak berubah (leakage aman).")

Validasi lulus: X_test & y_test tidak berubah (leakage aman).


## 3. Distribusi Kelas Sesudah SMOTE-ENN

In [5]:
print("Tipe hasil:", type(X_train_resampled).__name__, type(y_train_resampled).__name__)
print("Shape X_train_resampled:", X_train_resampled.shape)
print("Shape y_train_resampled:", np.asarray(y_train_resampled).shape)
print()
show_dist("y_train SESUDAH SMOTE-ENN", y_train_resampled)

Tipe hasil: DataFrame Series
Shape X_train_resampled: (34756, 4)
Shape y_train_resampled: (34756,)

y_train SESUDAH SMOTE-ENN (total 34,756):
    Not Stunted (0): 17,218  (49.54%)
    Stunted (1): 17,538  (50.46%)



## 4. Simpan Hasil Resampling

In [6]:
dump(X_train_resampled, os.path.join(DATA_DIR, "X_train_resampled.joblib"))
dump(y_train_resampled, os.path.join(DATA_DIR, "y_train_resampled.joblib"))

print("Tersimpan:")
print("  -", os.path.join(DATA_DIR, "X_train_resampled.joblib"))
print("  -", os.path.join(DATA_DIR, "y_train_resampled.joblib"))

Tersimpan:
  - ../data/processed\X_train_resampled.joblib
  - ../data/processed\y_train_resampled.joblib


## Ringkasan

- SMOTE-ENN diterapkan pada `X_train`/`y_train` saja.
- `X_test`/`y_test` tetap murni (divalidasi dengan `assert`).
- Hasil disimpan sebagai `X_train_resampled.joblib` & `y_train_resampled.joblib`.
- Notebook berikutnya melatih XGBoost & LightGBM pada data resampled ini,
  lalu dievaluasi pada test set yang sama dengan baseline.